In [ ]:
import os

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from specio import specread

In [ ]:
sp_folder_path  = '../0_initial_sp_files'
csv_folder_path = './csv_files'

ref_spectra_T      = specread('reference.sp')
ref_wavelength_T   =  ref_spectra_T.wavelength

spectra_all = os.listdir(sp_folder_path)

In [ ]:
for sp_filename in tqdm(spectra_all):
    try:
        file_path_sp = os.path.join(sp_folder_path, sp_filename)

        spectra_T = specread(file_path_sp)
        wavelength_T = spectra_T.wavelength
        intensity_T = spectra_T.amplitudes

        with np.errstate(divide='raise', invalid='raise'):
            try:
                
                wavelength_A = wavelength_T
                intensity_A = 2 - np.log10(intensity_T)

                if np.array_equal(wavelength_A, ref_wavelength_T):
                    data = {'Wavelength': wavelength_A, 'Absorbance': intensity_A}
                    df = pd.DataFrame(data)
                    csv_filename = sp_filename.replace('.sp', '.csv')
                    file_path_csv = os.path.join(csv_folder_path, csv_filename)
                    df.to_csv(file_path_csv, index=False)
                else:
                    print(f'Problem with wavelength correspondence in {file_path_sp}')
  
            except FloatingPointError:
                print(f'FloatingPointError during calculation in {file_path_sp}')

    except UnicodeDecodeError:
        print(f'Problem with encoding {file_path_sp}')